# 03 — Explore P-5 Organization Report Database

**Phase 1 — Data Discovery and Understanding**

This notebook is exploratory only. It answers:

- What does one row represent?
- How many records exist?
- What fields exist, and what are their data types?
- Are there missing values?
- What date range is covered?
- Are there duplicate records?
- Are there invalid values?

Source file: `data/raw/organizations/orf850.ebc` — an EBCDIC-encoded,
hierarchical mainframe extract of RRC's Organization Report ("P-5") tape,
downloaded from
https://www.rrc.texas.gov/resource-center/research/data-sets-available-for-download/.
Format is documented in
`data/raw/organizations/ora001_p5_manual_october-2014.pdf` and summarized in
`../docs/data_p5_organizations.md`; read that first for the full segment layout.

Key facts recap:
- Fixed **350-byte** binary records, EBCDIC encoded (assumed code page
  037 — this time **validated directly against decoded data**, see
  `../docs/data_p5_organizations.md`: real company names decode cleanly)
- No delimiters — records must be read in fixed strides
- **9 possible record types** — a much shallower hierarchy than either
  the production tape (24) or the P-4 tape (30) — identified by a
  **2-character alphanumeric key** (e.g. `'A '`, `'F '`), unlike the
  numeric keys on the other two tapes
- This notebook focuses on the **Organization record (`'A '`)** only —
  it carries the operator/organization number and the company name in
  one record, no further joins needed within this file
- The raw file is **214 MB / ~611K records** — much smaller than either
  the production tape or the P-4 tape, so this notebook can afford to do
  a genuine full-file scan for every question, no sampling needed.

In [12]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path("..") / "src"))

from oil_pipeline.extract.p5_organizations import RECORD_LENGTH, load_orf850

RAW_DATA_PATH = Path("..") / "data" / "raw" / "organizations" / "orf850.ebc"
RAW_DATA_PATH.resolve()


WindowsPath('C:/texas-oil-data-platform/data/raw/organizations/orf850.ebc')

## File size & record count

Confirm the file divides evenly into 350-byte records (validates the fixed-length assumption from the format spec) before reading anything.

In [13]:
file_size = RAW_DATA_PATH.stat().st_size
total_records, remainder = divmod(file_size, RECORD_LENGTH)

print(f"File size:        {file_size:,} bytes")
print(f"Record length:    {RECORD_LENGTH} bytes")
print(f"Total records:    {total_records:,}")
print(f"Remainder bytes:  {remainder} (should be 0 for a clean fixed-length file)")


File size:        213,886,400 bytes
Record length:    350 bytes
Total records:    611,104
Remainder bytes:  0 (should be 0 for a clean fixed-length file)


## What does one row represent?

One physical 350-byte record is **one record of one of 9 possible types**
(see `../docs/data_p5_organizations.md`), not one organization. The file
reading and EBCDIC decoding logic lives in `src/oil_pipeline/`
(`extract/p5_organizations.py`, `utils.py`), not in this notebook —
`load_orf850` below streams the entire file once, tallying every record
by its 2-character type key and parsing the Organization (`'A '`) records
as it goes.

## Scan the full file

`load_orf850` (in `src/oil_pipeline/extract/p5_organizations.py`) streams
all ~611K records in a single pass: tallies every record by type key, and
parses Organization records into a DataFrame. This file is small enough
(214 MB) that this finishes in well under a minute.

In [14]:
import logging

logging.basicConfig(level=logging.INFO, format="%(message)s")

results = load_orf850(RAW_DATA_PATH)

key_counts = results["key_counts"]
df_org = results["organizations"]

assert key_counts["count"].sum() == total_records, "scanned count should match the file-size-derived total"


Done: 611,104 records scanned, 74,947 Organization records found


In [15]:
key_counts


,key,count,segment,pct
0,P,337829,Remarks (RRC internal use),55.28
1,K,172071,Officer/Agent Information,28.16
2,A,74947,Organization Information (Root),12.26
3,U,10638,Activity Indicators,1.74
4,F,7799,Specialty Codes (specialty mailing addresses),1.28
5,R,6801,Activity Restrictions,1.11
6,H,980,Specialty Addresses by District,0.16
7,1T,39,Specialty/Activity Code Table (not org-specific),0.01


In [16]:
print(f"Organization ('A '): {len(df_org):,} rows")


Organization ('A '): 74,947 rows


## What fields exist, and what are their data types?

Preview the parsed Organization records. All columns come out as Python
`str` from the parser; `.info()` shows the resulting pandas dtypes.

In [17]:
df_org.head()


,operator_number,organization_name,p5_status
0,000016,"A & E MINERALS, LLC",A
1,000017,A + INVESTMENTS A TEXAS LLC,A
2,000020,"A & P OIL AND GAS COMPANY, LLC",A
3,000025,"A & A ENTERPRISES, L.L.C.",I
4,000036,A&D OILFIELD SERVICES LLC,D


In [18]:
df_org.info()


<class 'pandas.DataFrame'>
RangeIndex: 74947 entries, 0 to 74946
Data columns (total 3 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   operator_number    74947 non-null  str  
 1   organization_name  74947 non-null  str  
 2   p5_status          74947 non-null  str  
dtypes: str(3)
memory usage: 3.7 MB


## Are there missing values?

This is a fixed-format binary extract — every field is present in every
record by construction (no free-form nulls). "Missing" here instead means
a blank (all-space, stripped to empty string) `organization_name`.

In [19]:
print("Null counts (should all be 0 — fixed binary layout has no free-form nulls):")
print(df_org.isna().sum())
print()
blank_names = (df_org["organization_name"] == "").sum()
print(f"Organization records with blank organization_name: {blank_names:,} / {len(df_org):,}")


Null counts (should all be 0 — fixed binary layout has no free-form nulls):
operator_number      0
organization_name    0
p5_status            0
dtype: int64

Organization records with blank organization_name: 0 / 74,947


## What date range is covered?

`parse_organization` doesn't include `OROR-DATE-BUILT` (not needed for the
operator-number → company-name join). Per `../docs/data_p5_organizations.md`,
it sits at byte position 243–250 (1-indexed, `CCYYMMDD`).

This is exploratory-only code, not part of the reusable pipeline in
`src/oil_pipeline/`, so it's written directly here using the shared
`iter_records`/`decode_text` primitives. Since this file is small (214 MB),
this is a genuine full-file scan, not a sample.

In [20]:
from oil_pipeline.utils import decode_text, iter_records

ORGANIZATION_KEY = "A "

date_built_years = []
for record in iter_records(RAW_DATA_PATH, RECORD_LENGTH):
    if decode_text(record[0:2]) == ORGANIZATION_KEY:
        date_built_years.append(decode_text(record[242:246]))

date_built_years = pd.Series(date_built_years)
print(f"distinct OROR-DATE-BUILT years: {date_built_years.nunique():,}")
print(f"earliest: {date_built_years.min()}   latest: {date_built_years.max()}")
print()
print(date_built_years.value_counts().sort_index())


distinct OROR-DATE-BUILT years: 55
earliest: 0000   latest: 2021

0000    5939
1931       1
1968       1
1970    2068
1971     135
1972      41
1973       3
1974       1
1975       4
1976       1
1977    4250
1978    4170
1979    4361
1980    2493
1981    3216
1982    3033
1983    3076
1984    3143
1985    2856
1986    2326
1987    2204
1988    1841
1989    1713
1990    1681
1991    1317
1992    1000
1993     963
1994     928
1995     850
1996     825
1997     806
1998     740
1999     622
2000     670
2001     613
2002     551
2003     519
2004     604
2005     814
2006     849
2007     878
2008     849
2009     733
2010     768
2011    1155
2012    1410
2013    1171
2014    1140
2015     911
2016     799
2017     887
2018    1010
2019     907
2020     650
2021     451
Name: count, dtype: int64


## Are there duplicate records?

`operator_number` should be unique across Organization records — one `'A '`
record per organization.

In [21]:
dup_operators = df_org.duplicated(subset=["operator_number"]).sum()
print(f"Duplicate operator_number values: {dup_operators:,} / {len(df_org):,}")

exact_dup_org_rows = df_org.duplicated().sum()
print(f"Fully duplicate Organization rows: {exact_dup_org_rows:,}")


Duplicate operator_number values: 0 / 74,947
Fully duplicate Organization rows: 0


## Are there invalid values?

Sanity-check values against what the format spec says they should be:
- `operator_number` should be 6 numeric digits
- `p5_status` is documented as `A`/`I`/`D`/`S` only

In [22]:
DOCUMENTED_P5_STATUSES = {"A", "I", "D", "S"}

invalid_operator_number = (~df_org["operator_number"].str.fullmatch(r"\d{6}")).sum()
invalid_p5_status = (~df_org["p5_status"].isin(DOCUMENTED_P5_STATUSES)).sum()

print(f"Organization — operator_number not 6 digits:        {invalid_operator_number:,} / {len(df_org):,}")
print(f"Organization — p5_status not in {sorted(DOCUMENTED_P5_STATUSES)}: {invalid_p5_status:,} / {len(df_org):,}")
print()
print("Full p5_status distribution (including any undocumented values):")
print(df_org["p5_status"].value_counts())


Organization — operator_number not 6 digits:        0 / 74,947
Organization — p5_status not in ['A', 'D', 'I', 'S']: 555 / 74,947

Full p5_status distribution (including any undocumented values):
p5_status
I    63393
A     6829
D     3923
X      549
S      247
H        4
R        2
Name: count, dtype: int64


## Summary & next steps

Full-file scan of all 611,104 records:

- One physical row = one 350-byte record of 1 of 9 types, not one
  organization. Far fewer segment types than either the production tape
  (24) or P-4 (30) — a much shallower hierarchy.
- Organization (`'A '`): **74,947 records** — 12.3% of the file. The bulk
  of the file is `P ` (Remarks, 55.3%) and `K ` (Officer/Agent info,
  28.2%), both of which can repeat many times per organization.
- No null values are possible in this fixed binary layout, and there are
  **zero blank organization names** — every organization has a real name.
- `operator_number` is a clean 1:1 key: **zero duplicates**, zero fully
  duplicate rows.
- `OROR-DATE-BUILT` spans 55 distinct years, 1931–2021 (5,939 records have
  an unset `0000` value — likely legacy records predating this field being
  tracked).
- **Found 3 undocumented `p5_status` values**: `X` (549), `H` (4), `R` (2)
  — none are in the source spec, which only documents `A`/`I`/`D`/`S`. `H`
  and `R` are rare enough that an earlier ~6,000-record sample only caught
  `X`; the full-file scan is what surfaced all three. Updated
  `../docs/data_p5_organizations.md` with this corrected finding.
- Zero invalid `operator_number` values (all 6 numeric digits).

The file-reading and decoding logic lives in
`src/oil_pipeline/extract/p5_organizations.py` (`load_orf850`) and
`src/oil_pipeline/utils.py` — matching the pattern from
`01_explore_rrc_production.ipynb`. Together with
`02_explore_p4_operators.ipynb`, this is the data behind the
`lease_operators` table (`oil_pipeline/transform.py`), which brings real
company names into the `oil_production_violations_view` and
`total_oil_production_by_lease_id_view` Looker Studio views.